# Prompt chaining & structured outputs using Ollama
This notebook implements a five-step LLM pipeline: extract entities, analyse claims, synthesise a short summary, format a Markdown brief, and validate via critique. It uses JSON for reliable machine handoffs and Markdown for the final deliverable.
---

## Imports & config
Imports: `ollama` for the local LLM client, and Python stdlib (`json`, `textwrap`, `pathlib`, `datetime`) for parsing, logging, and file I/O. Set `MODEL`, `ARTICLE_PATH`, and `OUTPUT_PATH` at the top for easy configuration.

In [ ]:
import ollama  # Ollama client
import json    # JSON parsing
import textwrap  # Wrap long logs
from pathlib import Path  # File paths
from datetime import datetime  # Timestamp

# Config: model and paths
MODEL = "llama3"
ARTICLE_PATH = "sample_article.txt"
OUTPUT_PATH = "research_brief.md"

print(f"✅ Libraries loaded. Using model: {MODEL}")

✅ Libraries loaded. Using model: llama3


## Load article
Read the source article once into `article_text` to use as the single source of truth for all pipeline steps. This avoids repeated I/O and ensures consistent input for each LLM call.

In [ ]:
def load_article(path: str) -> str:
    "Load article text from disk.",
    return Path(path).read_text(encoding="utf-8")

# Load and preview (short)
article_text = load_article(ARTICLE_PATH)
print("📰 Article loaded — first 300 characters:")
print("-" * 60)
print(article_text[:300])
print(f"\n... [{len(article_text)} total characters]")

📰 Article loaded — first 300 characters:
------------------------------------------------------------
The Manhattan Project

The Manhattan Project was a research and development undertaking during World War II that produced the first nuclear weapons. It was led by the United States with the support of the United Kingdom and Canada. From 1942 to 1946, the project was under the direction of Major Gene

... [3978 total characters]


## Helpers: LLM call, JSON parser, logger
Centralised helpers provide consistent `chat` calls, strip markdown fences when parsing JSON, and print readable intermediate logs — keeping calls DRY and outputs auditable. Use a `system` role to enforce JSON-only responses where needed.

In [ ]:
def call_llm(user_prompt: str, system_prompt: str = "") -> str:
    "Send prompt to Ollama and return text response.",
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    response = ollama.chat(model=MODEL, messages=messages)
    return response.message.content

def parse_json_response(raw: str) -> object:
    "Parse JSON from LLM, stripping fences.",
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        cleaned = "\n".join(lines[1:-1]).strip()
    return json.loads(cleaned)

def log_step(step_num: int, title: str, output: str) -> None:
    "Print a short labelled step output.",
    border = "═" * 65
    print(f"\n{border}")
    print(f"  STEP {step_num}: {title}")
    print(border)
    for line in output.splitlines():
        print(textwrap.fill(line, width=70) if len(line) > 70 else line)
    print()

print("✅ Helper functions defined: call_llm(), parse_json_response(), log_step()")

✅ Helper functions defined: call_llm(), parse_json_response(), log_step()


## Step 1 — Extract entities (JSON)
Ask the model to return a JSON object with four lists: `people`, `organisations`, `dates`, and `locations`. Using JSON ensures reliable programmatic handoff to subsequent steps.

In [ ]:
def step1_extract(article: str) -> dict:
    "Extract named entities as JSON.",
    system = (
        "You are a precise named-entity extractor. ",
        "You ONLY output valid JSON — no prose, no markdown fences, no explanations.",
    )
    user = f"""Extract all named entities from the article below.

Return a single JSON object with exactly these four keys:
  "people"        : list of named individuals mentioned
  "organisations" : list of institutions, programmes, bodies
  "dates"         : list of specific dates, years, or time-ranges
  "locations"     : list of places, cities, or sites

Each list contains strings only. No nested objects.

ARTICLE:
{article}
"""
    raw_response = call_llm(user, system)
    entities = parse_json_response(raw_response)
    log_step(1, "EXTRACT — Named Entities (JSON)", json.dumps(entities, indent=2))
    return entities

# ── Run Step 1 ─────────────────────────────────────────────────────────────────
print("⏳ Running Step 1: Entity Extraction...")
entities = step1_extract(article_text)
print(f"✅ Step 1 complete. Found: {len(entities.get('people',[]))} people, ",
      f"{len(entities.get('organisations',[]))} orgs, ",
      f"{len(entities.get('dates',[]))} dates, ",
      f"{len(entities.get('locations',[]))} locations")

⏳ Running Step 1: Entity Extraction...

═════════════════════════════════════════════════════════════════
  STEP 1: EXTRACT — Named Entities (JSON)
═════════════════════════════════════════════════════════════════
{
  "people": [
    "Leslie Groves",
    "J. Robert Oppenheimer",
    "Enrico Fermi",
    "Niels Bohr",
    "Leo Szilard",
    "Klaus Fuchs",
    "Albert Einstein",
    "Franklin D. Roosevelt"
  ],
  "organisations": [
    "United States",
    "United Kingdom",
    "Canada",
    "U.S. Army Corps of Engineers",
    "Manhattan District",
    "Tube Alloys",
    "Atomic Energy Commission",
    "Hanford Site",
    "Los Alamos Laboratory",
    "Oak Ridge",
    "University of Chicago",
    "Metallurgical Laboratory"
  ],
  "dates": [
    "1939-1946",
    "1942",
    "1945",
    "1947",
    "1954",
    "August 6, 1945",
    "August 9, 1945",
    "August 15, 1945",
    "July 16, 1945",
    "December 2, 1942"
  ],
  "locations": [
    "Manhattan",
    "Hiroshima",
    "Nagasaki",
    "

## Step 2 — Analyse claims (JSON)
Using the article and extracted entities, identify the top three factual claims and assign a categorical confidence (`high`/`medium`/`low`) with brief justifications. Injecting Step 1's JSON makes reasoning and references explicit and consistent.

In [ ]:
def step2_analyse(article: str, entities: dict) -> list:
    "Identify top 3 claims with confidence.",
    system = (
        "You are a rigorous fact-analyst. ",
        "You ONLY output valid JSON — no prose, no markdown fences.",
    )
    user = f"""You are analysing a historical article. 
The following named entities have already been extracted:

,json.dumps(entities, indent=2),

From the article below, identify the 3 MOST IMPORTANT factual claims.

Return a JSON array of exactly 3 objects, each with:
  "claim"          : a concise 1-sentence statement of the claim
  "confidence"     : one of: "high", "medium", or "low"
  "justification"  : 1-2 sentences explaining the confidence rating


Confidence guide:
  high   = well-documented historical fact, cross-verifiable
  medium = plausible but contains estimates or contested details
  low    = speculative, anecdotal, or single-source

ARTICLE:
{article}
"""

⏳ Running Step 2: Claim Analysis...

═════════════════════════════════════════════════════════════════
  STEP 2: ANALYSE — Top 3 Claims with Confidence (JSON)
═════════════════════════════════════════════════════════════════
[
  {
    "claim": "The Manhattan Project was a research and development
undertaking during World War II that produced the first nuclear
weapons.",
    "confidence": "high",
    "justification": "Well-documented historical fact, cross-
verifiable through multiple sources."
  },
  {
    "claim": "J. Robert Oppenheimer led the scientific work on the
Manhattan Project.",
    "confidence": "high",
    "justification": "Oppenheimer is widely recognized as the leader
of the scientific effort during the project's development and
production phases."
  },
  {
    "claim": "The atomic bombings of Hiroshima and Nagasaki in Japan
on August 6 and 9, 1945 respectively led to Japan's unconditional
surrender on August 15, 1945.",
    "confidence": "high",
    "justification": "Wel

## Step 3 — Synthesise summary (prose)
Combine entities, claims, and the article to produce a ~200-word human-readable summary. Use prose here (not JSON) because the output is meant for people, not programmatic parsing.

In [ ]:
def step3_synthesise(article: str, entities: dict, claims: list) -> str:
    "Synthesize a ~200-word summary.",
    user = f"""You are a research analyst writing a concise briefing.

Using the information below, write a structured summary of approximately 200 words.
The summary must cover:
  1. What the article is about (context and significance)
  2. The key people and organisations involved
  3. The most important claims identified

Write in clear, professional prose. Do not use bullet points. Aim for ~200 words.

--- EXTRACTED ENTITIES ---
{json.dumps(entities, indent=2)}

--- KEY CLAIMS ---
{json.dumps(claims, indent=2)}

--- ORIGINAL ARTICLE ---
{article}
"""

    summary = call_llm(user)
    log_step(3, "SYNTHESISE — 200-Word Summary", summary)

    return summary

# ── Run Step 3 ─────────────────────────────────────────────────────────────────
print("⏳ Running Step 3: Synthesis...")
summary = step3_synthesise(article_text, entities, claims)
word_count = len(summary.split())
print(f"✅ Step 3 complete. Summary word count: {word_count}")

⏳ Running Step 3: Synthesis...

═════════════════════════════════════════════════════════════════
  STEP 3: SYNTHESISE — 200-Word Summary
═════════════════════════════════════════════════════════════════
Summary:

The article provides a comprehensive overview of the Manhattan
Project, a research and development undertaking during World War II
that produced the first nuclear weapons. Led by Major General Leslie
Groves and J. Robert Oppenheimer, the project was a collaborative
effort between the United States, the United Kingdom, and Canada,
involving over 130,000 people and costing nearly $2 billion. The
article highlights the significant scientific contributions of key
figures such as Enrico Fermi, Niels Bohr, Leo Szilard, and Klaus
Fuchs, among others.

The Manhattan Project resulted in the development of two types of
atomic bombs, Little Boy and Fat Man, which were used to devastating
effect in the bombings of Hiroshima and Nagasaki in August 1945. The
article notes that the bombings

## Step 4 — Format as Markdown brief
Take the summary, entities, and claims and produce a polished Markdown document with four sections: `Overview`, `Key Entities`, `Main Claims`, and `Confidence Assessment`. Separating content from presentation lets you change layout without re-running expensive LLM calls.

In [ ]:
def step4_format(summary: str, entities: dict, claims: list) -> str:
    "Format content into the required Markdown brief.",
    user = f"""Convert the research content below into a structured Markdown brief.

The brief MUST contain exactly these four sections with these exact headings:

## Overview
[Insert the 200-word summary here, unchanged]

## Key Entities
[List all entities grouped by type: People, Organisations, Dates, Locations]

## Main Claims
[List the 3 claims as numbered items with their confidence rating in parentheses]

## Confidence Assessment
[A short paragraph discussing the overall reliability of the claims and any caveats]

Use proper Markdown formatting. Do not add any section not listed above.

--- 200-WORD SUMMARY ---
{summary}
--- ENTITIES ---
{json.dumps(entities, indent=2)}

--- CLAIMS ---
{json.dumps(claims, indent=2)}
"""

    brief = call_llm(user)
    log_step(4, "FORMAT — Markdown Research Brief", brief)

    return brief

# ── Run Step 4 ─────────────────────────────────────────────────────────────────
print("⏳ Running Step 4: Markdown Formatting...")
brief = step4_format(summary, entities, claims)
print(f"✅ Step 4 complete. Brief length: {len(brief)} characters")

⏳ Running Step 4: Markdown Formatting...

═════════════════════════════════════════════════════════════════
  STEP 4: FORMAT — Markdown Research Brief
═════════════════════════════════════════════════════════════════
Here is the structured Markdown brief:

## Overview
The article provides a comprehensive overview of the Manhattan
Project, a research and development undertaking during World War II
that produced the first nuclear weapons. Led by Major General Leslie
Groves and J. Robert Oppenheimer, the project was a collaborative
effort between the United States, the United Kingdom, and Canada,
involving over 130,000 people and costing nearly $2 billion. The
article highlights the significant scientific contributions of key
figures such as Enrico Fermi, Niels Bohr, Leo Szilard, and Klaus
Fuchs, among others.

The Manhattan Project resulted in the development of two types of
atomic bombs, Little Boy and Fat Man, which were used to devastating
effect in the bombings of Hiroshima and Nagas

## Step 5 — Validate (critique)
Run a sceptical LLM persona to critique the final brief: flag unsupported claims, omissions, misclassifications, or inconsistent confidence ratings. This adversarial validation step improves reliability before saving the output.

In [ ]:
def step5_validate(brief: str, article: str) -> str:
    "Validate the brief and return a critique paragraph.",
    user = f"""You are a sceptical academic fact-checker reviewing a research brief.
Your job is to compare the brief against the original source article and identify:
  1. Any claims in the brief that are NOT supported by the article
  2. Any important facts from the article that were omitted
  3. Any confidence ratings that seem too high or too low given the evidence
  4. Any vague, ambiguous, or potentially misleading statements

Write a single critique paragraph (3-5 sentences) addressed to the brief's author.
Be specific — cite the problematic claim or entity by name.
If the brief is fully accurate, say so and explain why you're satisfied.

--- RESEARCH BRIEF TO VALIDATE ---
{brief}

--- ORIGINAL SOURCE ARTICLE ---
{article}
"""
    critique = call_llm(user)
    log_step(5, "VALIDATE — Critique Paragraph", critique)

    return critique

# ── Run Step 5 ─────────────────────────────────────────────────────────────────
print("⏳ Running Step 5: Validation & Critique...")
critique = step5_validate(brief, article_text)
print("✅ Step 5 complete. Critique generated.")

⏳ Running Step 5: Validation & Critique...

═════════════════════════════════════════════════════════════════
  STEP 5: VALIDATE — Critique Paragraph
═════════════════════════════════════════════════════════════════
I've thoroughly reviewed the research brief against the original
source article, and I'm pleased to report that the majority of the
brief accurately reflects the content and tone of the article.
However, I do have some concerns that require attention:

1. **Claim not supported by the article**: The brief states that "The
Manhattan Project was a comprehensive overview" (emphasis mine), which
is not entirely accurate. While the article does provide an overview,
it's more focused on detailing the project's history, key players, and
scientific achievements.
2. **Omitted fact**: The article mentions that Klaus Fuchs, a German-
born British theoretical physicist, worked on the project but was
later revealed to be a Soviet spy who had passed atomic secrets to the
USSR. This signif

## Assemble & save output
Combine the formatted brief and critique, add a timestamp, and write the result to `research_brief.md` so it can be reviewed, shared, and versioned without re-running the pipeline.

In [ ]:
def assemble_and_save(
    brief: str,
    critique: str,
    output_path: str
) -> str:
    "Assemble final markdown and save to disk.",
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    full_document = f"""# Research Brief\n\n> *Generated by a 5-step prompt chain using llama3 via Ollama*  \n> *Generated at: {timestamp}*\n\n---\n\n{brief}\n\n---\n\n## Validation Critique\n\n{critique}\n"""
    Path(output_path).write_text(full_document, encoding="utf-8")
    return full_document
# ── Assemble and save ─────────────────────────────────────────────────────────
final_document = assemble_and_save(brief, critique, OUTPUT_PATH)

print(f"💾 Research brief saved to: {OUTPUT_PATH}")
print(f"   Total document length: {len(final_document)} characters")
print(f"   Total word count:      {len(final_document.split())} words")

💾 Research brief saved to: research_brief.md
   Total document length: 4973 characters
   Total word count:      772 words


## Display final brief
Render the saved Markdown document in the notebook for an immediate, formatted view.

In [ ]:
from IPython.display import Markdown, display

# Render the saved research brief as formatted Markdown inside the notebook
display(Markdown(final_document))

# Research Brief

> *Generated by a 5-step prompt chain using llama3 via Ollama*  
> *Generated at: 2026-06-10 23:59:26*

---

Here is the structured Markdown brief:

## Overview
The article provides a comprehensive overview of the Manhattan Project, a research and development undertaking during World War II that produced the first nuclear weapons. Led by Major General Leslie Groves and J. Robert Oppenheimer, the project was a collaborative effort between the United States, the United Kingdom, and Canada, involving over 130,000 people and costing nearly $2 billion. The article highlights the significant scientific contributions of key figures such as Enrico Fermi, Niels Bohr, Leo Szilard, and Klaus Fuchs, among others.

The Manhattan Project resulted in the development of two types of atomic bombs, Little Boy and Fat Man, which were used to devastating effect in the bombings of Hiroshima and Nagasaki in August 1945. The article notes that the bombings led to Japan's unconditional surrender, bringing an end to World War II. Additionally, the project's Trinity test on July 16, 1945, marked a major milestone in nuclear history.

The article also touches on the legacy of the Manhattan Project, including its impact on the Cold War era and the subsequent development of international concerns about nuclear proliferation. Several key figures involved in the project, including Oppenheimer and Albert Einstein, later expressed moral reservations about the project's outcome and the consequences that followed.

Overall, this summary provides a concise overview of the Manhattan Project, highlighting its significance, key players, and lasting impact on the world.

## Key Entities
### People
* Leslie Groves
* J. Robert Oppenheimer
* Enrico Fermi
* Niels Bohr
* Leo Szilard
* Klaus Fuchs
* Albert Einstein
* Franklin D. Roosevelt

### Organisations
* United States
* United Kingdom
* Canada
* U.S. Army Corps of Engineers
* Manhattan District
* Tube Alloys
* Atomic Energy Commission
* Hanford Site
* Los Alamos Laboratory
* Oak Ridge
* University of Chicago
* Metallurgical Laboratory

### Dates
* 1939-1946
* 1942
* 1945
* 1947
* 1954
* August 6, 1945
* August 9, 1945
* August 15, 1945
* July 16, 1945
* December 2, 1942

### Locations
* Manhattan
* Hiroshima
* Nagasaki
* Japan
* New Mexico
* Washington state
* Tennessee

## Main Claims
1. "The Manhattan Project was a research and development undertaking during World War II that produced the first nuclear weapons." (confidence: high)
2. "J. Robert Oppenheimer led the scientific work on the Manhattan Project." (confidence: high)
3. "The atomic bombings of Hiroshima and Nagasaki in Japan on August 6 and 9, 1945 respectively led to Japan's unconditional surrender on August 15, 1945." (confidence: high)

## Confidence Assessment
While the claims presented are based on well-documented historical facts, some caveats should be considered. The Manhattan Project was a complex undertaking that involved many individuals and organisations, and while Oppenheimer is widely recognized as the leader of the scientific effort, other figures also played crucial roles. Additionally, the project's legacy extends beyond its immediate impact, with ongoing concerns about nuclear proliferation and the long-term consequences of the bombings.

---

## Validation Critique

I've thoroughly reviewed the research brief against the original source article, and I'm pleased to report that the majority of the brief accurately reflects the content and tone of the article. However, I do have some concerns that require attention:

1. **Claim not supported by the article**: The brief states that "The Manhattan Project was a comprehensive overview" (emphasis mine), which is not entirely accurate. While the article does provide an overview, it's more focused on detailing the project's history, key players, and scientific achievements.
2. **Omitted fact**: The article mentions that Klaus Fuchs, a German-born British theoretical physicist, worked on the project but was later revealed to be a Soviet spy who had passed atomic secrets to the USSR. This significant detail is missing from the brief.
3. **Confidence rating too high**: Claim 1 ("The Manhattan Project was a research and development undertaking during World War II that produced the first nuclear weapons.") should have a confidence rating of "medium" instead of "high", as there might be minor discrepancies in the article's account of the project's scope and timeline.
4. **Vague statement**: The brief states that the project's legacy extends beyond its immediate impact, with ongoing concerns about nuclear proliferation and the long-term consequences of the bombings. While this is true, the language is too vague to provide a clear understanding of the specific issues or implications.

Overall, the research brief provides a solid foundation for discussing the Manhattan Project, but some minor adjustments can improve its accuracy and clarity.


## One-shot vs chained generation
Compare a one-shot prompt that generates the full brief + critique with the chained pipeline to inspect differences in accuracy, structure, and critique quality. Use the validator step to pick the better output.

In [ ]:
import difflib
def generate_direct_document(article: str) -> str:
    "One-shot generate full brief + critique.",
    system = (
        "You are a research analyst who outputs a complete Markdown research brief with a ",
        "validation critique. Output only Markdown with no extra prose outside the required sections.",
    )
    user = f"""Write a final Markdown research brief and validation critique from the article below.

The document MUST include these sections exactly:

# Research Brief

## Overview
A concise ~200-word summary of the article.

## Key Entities
A grouped list of People, Organisations, Dates, Locations.

## Main Claims
A numbered list of the 3 most important factual claims, each with a confidence rating in parentheses.

## Confidence Assessment
A short paragraph discussing overall claim reliability.

## Validation Critique
A 3-5 sentence critique identifying any unsupported or weak claims, omitted facts, or overly confident ratings.

Do not include any sections other than the ones listed above.

ARTICLE:
{article}
"""
    response = call_llm(user, system)
    return response

def validate_comparison(article: str, chained: str, direct: str) -> dict:
    "Ask model to compare chained vs direct outputs.",
    system = (
        "You are a careful validation assistant. Compare two research briefs generated from the same source article. ",
        "Output only valid JSON with no markdown fences or extra explanation.",
    )
    user = f"""Compare the two generated documents below for the same source article.

SOURCE ARTICLE:
{article}

CHAINED OUTPUT:
{chained}

DIRECT OUTPUT:
{direct}
"""
    raw = call_llm(user, system)
    return parse_json_response(raw)

print("⏳ Running direct one-shot generation...")
direct_document = generate_direct_document(article_text)

print("✅ One-shot generation complete.")
print("\n--- Direct Output Preview ---\n")
print(direct_document[:1200])

print("\n--- Comparison with Chained Output ---\n")
print(f"Chained document length: {len(final_document)} characters")
print(f"Direct document length:  {len(direct_document)} characters")
print(f"Chained word count:      {len(final_document.split())}")
print(f"Direct word count:       {len(direct_document.split())}")

comparison = validate_comparison(article_text, final_document, direct_document)
print("\n--- LLM Validator Comparison ---\n")
print(json.dumps(comparison, indent=2))
# Show a useful similarity score rather than a raw unified diff.
similarity = difflib.SequenceMatcher(None, final_document, direct_document).ratio()
print(f"\nDocument similarity ratio: {similarity:.3f}")

⏳ Running direct one-shot generation...
✅ One-shot generation complete.

--- Direct Output Preview ---

# Research Brief

## Overview
The Manhattan Project was a research and development undertaking during World War II that produced the first nuclear weapons. Led by Major General Leslie Groves of the U.S. Army Corps of Engineers, the project involved over 130,000 people and cost nearly US$2 billion. The project absorbed its earlier British counterpart, Tube Alloys, and employed more than 30 sites across the United States, the United Kingdom, and Canada.

The Manhattan Project developed two types of atomic bombs: "Little Boy" (uranium gun-type fission weapon) and "Fat Man" (plutonium implosion-type nuclear weapon). The project also had a smaller research program aimed at developing a nuclear reactor, which culminated in the first artificial nuclear reactor, Chicago Pile-1. Scientific work was led by American physicist J. Robert Oppenheimer, with notable contributions from Enrico Fermi, 